In [1]:
import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU — Task 2 will be very slow")

cuda: True
device: Tesla T4


In [2]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

REPO_DIR = Path("/content/drive/MyDrive/MS AI/Semester_3/ATML/PAs/ATML-PA1")
os.chdir(REPO_DIR)
print("cwd:", Path.cwd())
print("task2 exists:", (REPO_DIR / "task2").is_dir())
print("train.py exists:", (REPO_DIR / "task2" / "train.py").is_file())
print("shared exists:", (REPO_DIR / "shared" / "pacs.py").is_file())

Mounted at /content/drive
cwd: /content/drive/MyDrive/MS AI/Semester_3/ATML/PAs/ATML-PA1
task2 exists: True
train.py exists: True
shared exists: True


In [3]:
%pip install -q -r requirements.txt gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.5 MB/s eta 0:00:00


In [5]:
from pathlib import Path
import shutil
import zipfile
import urllib.request

pacs_root = Path("data/pacs")
marker = pacs_root / "photo"

def count_domain(d):
    return sum(1 for p in (pacs_root / d).rglob("*") if p.is_file())

if marker.is_dir() and count_domain("photo") > 0:
    print("PACS already present at", pacs_root.resolve())
else:
    pacs_root.parent.mkdir(parents=True, exist_ok=True)
    zip_path = Path("data/PACS.zip")
    if zip_path.exists() and zip_path.stat().st_size < 1_000_000:
        zip_path.unlink()  # previous failed gdown stub

    if not zip_path.exists():
        url = "https://huggingface.co/datasets/Azeez577/PACS/resolve/main/PACS.zip"
        print("Downloading", url)
        urllib.request.urlretrieve(url, zip_path)
        print("Downloaded bytes:", zip_path.stat().st_size)

    extract_dir = Path("data/_pacs_extract")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

    # Find folder containing the four domains
    candidates = list(extract_dir.rglob("photo"))
    candidates = [c.parent for c in candidates if c.is_dir()]
    if not candidates:
        raise FileNotFoundError("Could not locate PACS photo/ after unzip; tree=" + str(list(extract_dir.rglob("*"))[:40]))
    src = candidates[0]
    print("Found domain root:", src)
    if pacs_root.exists():
        shutil.rmtree(pacs_root)
    shutil.move(str(src), str(pacs_root))
    print("Installed PACS ->", pacs_root.resolve())

for d in ["photo", "art_painting", "cartoon", "sketch"]:
    print(f"  {d}: {count_domain(d)} files")

Downloaded bytes: 184417365
Found domain root: data/_pacs_extract/PACS
Installed PACS -> /content/drive/MyDrive/MS AI/Semester_3/ATML/PAs/ATML-PA1/data/pacs
  photo: 1670 files
  art_painting: 2048 files
  cartoon: 2344 files
  sketch: 3929 files


In [ ]:
# Probe alternate PACS sources
import urllib.request
urls = [
    "https://huggingface.co/datasets/flwrlabs/pacs",
    "https://huggingface.co/api/datasets/flwrlabs/pacs",
]
for u in urls:
    try:
        r = urllib.request.urlopen(u, timeout=20)
        print(r.status, u[:80])
    except Exception as e:
        print("FAIL", type(e).__name__, u[:80], e)

In [8]:
!python -m shared.prepare_pacs_splits

PACS layout OK
  root: /content/drive/MyDrive/MS AI/Semester_3/ATML/PAs/ATML-PA1/data/pacs
  n_total: 9991
  photo: 1670
  art_painting: 2048
  cartoon: 2344
  sketch: 3929
Wrote splits -> /content/drive/MyDrive/MS AI/Semester_3/ATML/PAs/ATML-PA1/shared/splits/pacs_sketch_seed6304.json


In [7]:
# Ensure Drive has the ImageNet-mean fix (in case sync lags)
from pathlib import Path
p = Path("shared/pacs.py")
text = p.read_text(encoding="utf-8")
if 'IMAGENET_MEAN = (0.485' not in text:
    text = text.replace(
        "from torchvision import transforms\nfrom torchvision.models import ResNet18_Weights\n",
        "from torchvision import transforms\n",
    )
    old = (
        "# ImageNet normalization tied to ResNet18_Weights.IMAGENET1K_V1\n"
        "_IMAGENET_WEIGHTS = ResNet18_Weights.IMAGENET1K_V1\n"
        "IMAGENET_MEAN = _IMAGENET_WEIGHTS.meta[\"mean\"]\n"
        "IMAGENET_STD = _IMAGENET_WEIGHTS.meta[\"std\"]\n"
    )
    new = (
        "# ImageNet normalization for ResNet18_Weights.IMAGENET1K_V1.\n"
        "# Hardcoded because some torchvision builds omit mean/std from weights.meta.\n"
        "IMAGENET_MEAN = (0.485, 0.456, 0.406)\n"
        "IMAGENET_STD = (0.229, 0.224, 0.225)\n"
    )
    if old not in text:
        raise SystemExit("Could not patch shared/pacs.py automatically")
    p.write_text(text.replace(old, new), encoding="utf-8")
    print("Patched shared/pacs.py on Drive")
else:
    print("shared/pacs.py already has hardcoded ImageNet mean/std")
print(p.resolve())

shared/pacs.py already has hardcoded ImageNet mean/std
/content/drive/MyDrive/MS AI/Semester_3/ATML/PAs/ATML-PA1/shared/pacs.py


In [9]:
!python -m task2.train --method-config task2/configs/source_only.yaml

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 180MB/s]
Training source_only on cuda | steps/epoch=202
[source_only] epoch 01  loss=0.7861 cls=0.7861 align=0.0000  mean_src_f1=0.9000
[source_only] epoch 02  loss=0.4935 cls=0.4935 align=0.0000  mean_src_f1=0.8861
[source_only] epoch 03  loss=0.4150 cls=0.4150 align=0.0000  mean_src_f1=0.9254
[source_only] epoch 04  loss=0.3789 cls=0.3789 align=0.0000  mean_src_f1=0.9331
[source_only] epoch 05  loss=0.3615 cls=0.3615 align=0.0000  mean_src_f1=0.8989
[source_only] epoch 06  loss=0.3118 cls=0.3118 align=0.0000  mean_src_f1=0.9291
[source_only] epoch 07  loss=0.3105 cls=0.3105 align=0.0000  mean_src_f1=0.9060
[source_only] epoch 08  loss=0.2941 cls=0.2941 align=0.0000  mean_src_f1=0.9249
[source_only] epoch 09  loss=0.2636 cls=0.2636 align=0.0000  mean_src_f1=0.9129
Early stop at epoch 9 (best epoch 4)
Saved checkpoint -> tas

In [10]:
!python -m task2.train --method-config task2/configs/dan.yaml

Training dan_lambda1.0 on cuda | steps/epoch=202
[dan_lambda1.0] epoch 01  loss=0.7936 cls=0.7458 align=0.0478  mean_src_f1=0.9005
[dan_lambda1.0] epoch 02  loss=0.5141 cls=0.4933 align=0.0209  mean_src_f1=0.8987
[dan_lambda1.0] epoch 03  loss=0.4234 cls=0.4092 align=0.0142  mean_src_f1=0.9249
[dan_lambda1.0] epoch 04  loss=0.3872 cls=0.3733 align=0.0139  mean_src_f1=0.9183
[dan_lambda1.0] epoch 05  loss=0.3883 cls=0.3707 align=0.0176  mean_src_f1=0.9162
[dan_lambda1.0] epoch 06  loss=0.3754 cls=0.3492 align=0.0262  mean_src_f1=0.9343
[dan_lambda1.0] epoch 07  loss=0.3309 cls=0.3166 align=0.0143  mean_src_f1=0.9316
[dan_lambda1.0] epoch 08  loss=0.2875 cls=0.2753 align=0.0121  mean_src_f1=0.9380
[dan_lambda1.0] epoch 09  loss=0.2911 cls=0.2778 align=0.0133  mean_src_f1=0.9189
[dan_lambda1.0] epoch 10  loss=0.2688 cls=0.2566 align=0.0122  mean_src_f1=0.8905
[dan_lambda1.0] epoch 11  loss=0.2916 cls=0.2837 align=0.0079  mean_src_f1=0.9397
[dan_lambda1.0] epoch 12  loss=0.2824 cls=0.2740 

In [11]:
!python -m task2.train --method-config task2/configs/dann.yaml

Training dann on cuda | steps/epoch=202
[dann] epoch 01  loss=547.9946 cls=28.5437 align=519.4510  mean_src_f1=0.0381
[dann] epoch 02  loss=14.1727 cls=3.6432 align=10.5295  mean_src_f1=0.0505
[dann] epoch 03  loss=3.7685 cls=2.2933 align=1.4752  mean_src_f1=0.0923
[dann] epoch 04  loss=4.2112 cls=2.3274 align=1.8838  mean_src_f1=0.0992
[dann] epoch 05  loss=3.3915 cls=2.2054 align=1.1861  mean_src_f1=0.0494
[dann] epoch 06  loss=4.5512 cls=2.5657 align=1.9855  mean_src_f1=0.0705
[dann] epoch 07  loss=3.1465 cls=2.2082 align=0.9383  mean_src_f1=0.0513
[dann] epoch 08  loss=2.5957 cls=1.9337 align=0.6620  mean_src_f1=0.0519
[dann] epoch 09  loss=2.5799 cls=1.8948 align=0.6852  mean_src_f1=0.0641
Early stop at epoch 9 (best epoch 4)
Saved checkpoint -> task2/results/checkpoints/dann_best.pt
Saved curves     -> task2/results/curves/dann_history.json


In [12]:
import torch, torch.nn.functional as F
from pathlib import Path
import yaml
from task2.models.backbone import ResNet18Classifier
from task2.methods.dann import DANNMethod
from task2.train import build_dataloaders, load_merged_config
from shared.pacs import SOURCE_DOMAINS
from shared.pacs_protocol import cyclic_loader, set_train_mode_with_frozen_bn, unpack_batch
from task2.models.grl import grl_lambda

cfg = load_merged_config(Path('task2/configs/base.yaml'), Path('task2/configs/dann.yaml'))
device = torch.device('cuda')
loaders = build_dataloaders(cfg)
model = ResNet18Classifier().to(device)
method = DANNMethod(feature_dim=512).to(device)
set_train_mode_with_frozen_bn(model)
method.train()

src_cyc = {d: cyclic_loader(loaders['source_train'][d]) for d in SOURCE_DOMAINS}
t_cyc = cyclic_loader(loaders['target_train'])

source_images, source_labels = [], []
for d in SOURCE_DOMAINS:
    imgs, labels, _ = unpack_batch(next(src_cyc[d]), device)
    source_images.append(imgs); source_labels.append(labels)
source_images = torch.cat(source_images); source_labels = torch.cat(source_labels)
timgs, _, _ = unpack_batch(next(t_cyc), device)

sf, sl = model(source_images)
tf, tl = model(timgs)
print('feat mean/std/max', sf.mean().item(), sf.std().item(), sf.abs().max().item())
print('logits CE', F.cross_entropy(sl, source_labels).item())
print('alpha@0', grl_lambda(0.0), 'alpha@0.03', grl_lambda(0.03), 'alpha@0.5', grl_lambda(0.5))

opt = torch.optim.AdamW(list(model.parameters())+list(method.parameters()), lr=1e-4, weight_decay=1e-4)
for step in range(5):
    progress = step / (30*202)
    set_train_mode_with_frozen_bn(model)
    source_images, source_labels = [], []
    for d in SOURCE_DOMAINS:
        imgs, labels, _ = unpack_batch(next(src_cyc[d]), device)
        source_images.append(imgs); source_labels.append(labels)
    source_images = torch.cat(source_images); source_labels = torch.cat(source_labels)
    timgs, _, _ = unpack_batch(next(t_cyc), device)
    sf, sl = model(source_images)
    tf, _ = model(timgs)
    out = method.total_loss(source_logits=sl, source_labels=source_labels, source_features=sf, target_features=tf, progress=progress)
    opt.zero_grad(); out['loss'].backward(); opt.step()
    print(f'step {step} progress={progress:.4f} alpha={method.grl.alpha:.4f} cls={out["cls_loss"].item():.4f} dom={out["align_loss"].item():.4f} feat_max={sf.abs().max().item():.2f}')

feat mean/std/max 0.8467992544174194 0.8636674284934998 7.53076696395874
logits CE 2.1321773529052734
alpha@0 0.0 alpha@0.03 0.14888501167297363 alpha@0.5 0.9866143465042114
step 0 progress=0.0000 alpha=0.0000 cls=2.2334 dom=0.6793 feat_max=7.46
step 1 progress=0.0002 alpha=0.0008 cls=2.0346 dom=0.6190 feat_max=6.20
step 2 progress=0.0003 alpha=0.0017 cls=1.8820 dom=0.6593 feat_max=5.55
step 3 progress=0.0005 alpha=0.0025 cls=2.0167 dom=0.6794 feat_max=8.18
step 4 progress=0.0007 alpha=0.0033 cls=1.8695 dom=0.6782 feat_max=6.29


In [13]:
# Longer DANN stability probe (100 steps ~= half epoch)
import torch, torch.nn.functional as F
from pathlib import Path
from task2.models.backbone import ResNet18Classifier
from task2.methods.dann import DANNMethod
from task2.train import build_dataloaders, load_merged_config
from shared.pacs import SOURCE_DOMAINS
from shared.pacs_protocol import cyclic_loader, set_train_mode_with_frozen_bn, unpack_batch

cfg = load_merged_config(Path('task2/configs/base.yaml'), Path('task2/configs/dann.yaml'))
device = torch.device('cuda')
loaders = build_dataloaders(cfg)
model = ResNet18Classifier().to(device)
method = DANNMethod(feature_dim=512).to(device)
opt = torch.optim.AdamW(list(model.parameters())+list(method.parameters()), lr=1e-4, weight_decay=1e-4)
src_cyc = {d: cyclic_loader(loaders['source_train'][d]) for d in SOURCE_DOMAINS}
t_cyc = cyclic_loader(loaders['target_train'])
max_epochs, steps = 30, loaders['steps_per_epoch']

for step in range(100):
    progress = step / (max_epochs * steps)
    set_train_mode_with_frozen_bn(model); method.train()
    source_images, source_labels = [], []
    for d in SOURCE_DOMAINS:
        imgs, labels, _ = unpack_batch(next(src_cyc[d]), device)
        source_images.append(imgs); source_labels.append(labels)
    source_images = torch.cat(source_images); source_labels = torch.cat(source_labels)
    timgs, _, _ = unpack_batch(next(t_cyc), device)
    sf, sl = model(source_images)
    tf, _ = model(timgs)
    out = method.total_loss(source_logits=sl, source_labels=source_labels, source_features=sf, target_features=tf, progress=progress)
    opt.zero_grad(); out['loss'].backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), float('inf'))
    opt.step()
    if step % 20 == 0 or out['align_loss'].item() > 5:
        print(f'step {step:3d} alpha={method.grl.alpha:.4f} cls={out["cls_loss"].item():.3f} dom={out["align_loss"].item():.3f} grad_norm={float(grad_norm):.2f} feat_max={sf.detach().abs().max().item():.2f}')
print('done')

step   0 alpha=0.0000 cls=2.018 dom=0.704 grad_norm=11.55 feat_max=11.80
step  20 alpha=0.0165 cls=1.001 dom=0.425 grad_norm=14.85 feat_max=17.61
step  40 alpha=0.0330 cls=1.088 dom=0.378 grad_norm=16.14 feat_max=10.16
step  60 alpha=0.0495 cls=0.405 dom=0.311 grad_norm=9.00 feat_max=17.26
step  80 alpha=0.0659 cls=0.476 dom=0.376 grad_norm=14.04 feat_max=21.23
done


In [14]:
!python -m task2.train --method-config task2/configs/dann.yaml

Training dann on cuda | steps/epoch=202
[dann] epoch 01  loss=547.9946 cls=28.5437 align=519.4510  mean_src_f1=0.0381
[dann] epoch 02  loss=14.1727 cls=3.6432 align=10.5295  mean_src_f1=0.0505
[dann] epoch 03  loss=3.7685 cls=2.2933 align=1.4752  mean_src_f1=0.0923
[dann] epoch 04  loss=4.2112 cls=2.3274 align=1.8838  mean_src_f1=0.0992
[dann] epoch 05  loss=3.3915 cls=2.2054 align=1.1861  mean_src_f1=0.0494
[dann] epoch 06  loss=4.5512 cls=2.5657 align=1.9855  mean_src_f1=0.0705
[dann] epoch 07  loss=3.1465 cls=2.2082 align=0.9383  mean_src_f1=0.0513
[dann] epoch 08  loss=2.5957 cls=1.9337 align=0.6620  mean_src_f1=0.0519
[dann] epoch 09  loss=2.5799 cls=1.8948 align=0.6852  mean_src_f1=0.0641
Early stop at epoch 9 (best epoch 4)
Saved checkpoint -> task2/results/checkpoints/dann_best.pt
Saved curves     -> task2/results/curves/dann_history.json


In [15]:
# Reproduce DANN explosion with seed 6304
import torch
from pathlib import Path
from common.seed import set_seed
from task2.models.backbone import ResNet18Classifier
from task2.methods.dann import DANNMethod
from task2.train import build_dataloaders, load_merged_config, build_method
from shared.pacs import SOURCE_DOMAINS
from shared.pacs_protocol import cyclic_loader, set_train_mode_with_frozen_bn, unpack_batch

set_seed(6304)
cfg = load_merged_config(Path('task2/configs/base.yaml'), Path('task2/configs/dann.yaml'))
device = torch.device('cuda')
loaders = build_dataloaders(cfg)
model = ResNet18Classifier().to(device)
method = build_method(cfg).to(device)
opt = torch.optim.AdamW(list(model.parameters())+list(method.parameters()), lr=1e-4, weight_decay=1e-4)
src_cyc = {d: cyclic_loader(loaders['source_train'][d]) for d in SOURCE_DOMAINS}
t_cyc = cyclic_loader(loaders['target_train'])
max_epochs, steps = 30, loaders['steps_per_epoch']
print('method', type(method).__name__, 'lambda', method.lambda_domain)

for step in range(202):
    progress = ((1 - 1) * steps + step) / (max_epochs * steps)
    set_train_mode_with_frozen_bn(model); method.train()
    source_images, source_labels = [], []
    for d in SOURCE_DOMAINS:
        imgs, labels, _ = unpack_batch(next(src_cyc[d]), device)
        source_images.append(imgs); source_labels.append(labels)
    source_images = torch.cat(source_images); source_labels = torch.cat(source_labels)
    timgs, _, _ = unpack_batch(next(t_cyc), device)
    sf, sl = model(source_images)
    tf, _ = model(timgs)
    out = method.total_loss(source_logits=sl, source_labels=source_labels, source_features=sf, target_features=tf, progress=progress)
    opt.zero_grad(); out['loss'].backward()
    gn = torch.nn.utils.clip_grad_norm_(list(model.parameters())+list(method.parameters()), float('inf'))
    opt.step()
    if step < 5 or step % 25 == 0 or out['align_loss'].item() > 2:
        print(f'step {step:3d} alpha={method.grl.alpha:.4f} cls={out["cls_loss"].item():.3f} dom={out["align_loss"].item():.3f} grad={float(gn):.1f} feat_max={sf.detach().abs().max().item():.1f}')
        if out['align_loss'].item() > 50:
            print('EXPLODED');
            break
print('epoch1 done probe')

method DANNMethod lambda 1.0
step   0 alpha=0.0000 cls=2.132 dom=0.762 grad=13.4 feat_max=7.4
step   1 alpha=0.0008 cls=2.043 dom=0.700 grad=11.7 feat_max=9.5
step   2 alpha=0.0017 cls=1.739 dom=0.711 grad=10.0 feat_max=6.6
step   3 alpha=0.0025 cls=1.795 dom=0.667 grad=9.7 feat_max=6.3
step   4 alpha=0.0033 cls=1.646 dom=0.695 grad=7.1 feat_max=6.8
step  25 alpha=0.0206 cls=0.872 dom=0.464 grad=13.3 feat_max=16.6
step  50 alpha=0.0412 cls=0.665 dom=0.348 grad=10.5 feat_max=15.4
step  75 alpha=0.0618 cls=0.566 dom=0.342 grad=8.1 feat_max=13.9
step 100 alpha=0.0823 cls=0.695 dom=0.480 grad=11.9 feat_max=17.3
step 125 alpha=0.1028 cls=0.786 dom=1.463 grad=22.0 feat_max=17.2
step 129 alpha=0.1060 cls=0.508 dom=2.526 grad=19.7 feat_max=19.5
step 130 alpha=0.1069 cls=1.211 dom=2.336 grad=25.9 feat_max=20.6
step 131 alpha=0.1077 cls=0.417 dom=2.764 grad=23.3 feat_max=28.5
step 132 alpha=0.1085 cls=0.779 dom=3.039 grad=28.8 feat_max=20.2
step 133 alpha=0.1093 cls=0.902 dom=3.108 grad=29.8 fea

In [16]:
# Force-sync stability fixes onto Drive copies if needed, then probe
from pathlib import Path

# Patch discriminator init if missing
p = Path('task2/models/domain_discriminator.py')
t = p.read_text()
if 'std=0.01' not in t:
    raise SystemExit('discriminator patch not synced yet')
print('discriminator OK')

# Patch train clip if missing
pt = Path('task2/train.py').read_text()
if 'grad_clip_norm' not in pt:
    raise SystemExit('train.py clip not synced yet')
print('train clip OK')

import torch
from common.seed import set_seed
from task2.models.backbone import ResNet18Classifier
from task2.train import build_dataloaders, load_merged_config, build_method
from shared.pacs import SOURCE_DOMAINS
from shared.pacs_protocol import cyclic_loader, set_train_mode_with_frozen_bn, unpack_batch

set_seed(6304)
cfg = load_merged_config(Path('task2/configs/base.yaml'), Path('task2/configs/dann.yaml'))
print('grad_clip_norm', cfg['training'].get('grad_clip_norm'))
device = torch.device('cuda')
loaders = build_dataloaders(cfg)
model = ResNet18Classifier().to(device)
method = build_method(cfg).to(device)
params = list(model.parameters()) + list(method.parameters())
opt = torch.optim.AdamW(params, lr=1e-4, weight_decay=1e-4)
src_cyc = {d: cyclic_loader(loaders['source_train'][d]) for d in SOURCE_DOMAINS}
t_cyc = cyclic_loader(loaders['target_train'])
max_epochs, steps = 30, loaders['steps_per_epoch']
max_dom = 0
for step in range(202):
    progress = step / (max_epochs * steps)
    set_train_mode_with_frozen_bn(model); method.train()
    source_images, source_labels = [], []
    for d in SOURCE_DOMAINS:
        imgs, labels, _ = unpack_batch(next(src_cyc[d]), device)
        source_images.append(imgs); source_labels.append(labels)
    source_images = torch.cat(source_images); source_labels = torch.cat(source_labels)
    timgs, _, _ = unpack_batch(next(t_cyc), device)
    sf, sl = model(source_images)
    tf, _ = model(timgs)
    out = method.total_loss(source_logits=sl, source_labels=source_labels, source_features=sf, target_features=tf, progress=progress)
    opt.zero_grad(); out['loss'].backward()
    torch.nn.utils.clip_grad_norm_(params, 20.0)
    opt.step()
    max_dom = max(max_dom, out['align_loss'].item())
    if step in (0,50,100,150,201) or out['align_loss'].item()>5:
        print(f'step {step:3d} alpha={method.grl.alpha:.3f} cls={out["cls_loss"].item():.3f} dom={out["align_loss"].item():.3f} feat_max={sf.detach().abs().max().item():.1f}')
print('max_dom_epoch1', max_dom)

discriminator OK
train clip OK
grad_clip_norm 20.0
step   0 alpha=0.000 cls=2.132 dom=0.762 feat_max=7.4
step  50 alpha=0.041 cls=0.673 dom=0.343 feat_max=15.2
step 100 alpha=0.082 cls=0.692 dom=0.468 feat_max=19.0
step 135 alpha=0.111 cls=0.978 dom=5.730 feat_max=27.7
step 136 alpha=0.112 cls=0.989 dom=6.415 feat_max=57.1
step 137 alpha=0.113 cls=0.593 dom=6.908 feat_max=40.2
step 138 alpha=0.113 cls=0.896 dom=6.712 feat_max=40.7
step 139 alpha=0.114 cls=1.437 dom=8.685 feat_max=77.7
step 140 alpha=0.115 cls=0.966 dom=8.475 feat_max=76.0
step 141 alpha=0.116 cls=1.201 dom=7.508 feat_max=88.6
step 142 alpha=0.117 cls=1.118 dom=6.810 feat_max=120.9
step 143 alpha=0.117 cls=0.827 dom=6.233 feat_max=102.8
step 144 alpha=0.118 cls=0.863 dom=5.433 feat_max=114.0
step 149 alpha=0.122 cls=0.398 dom=5.046 feat_max=77.9
step 150 alpha=0.123 cls=1.003 dom=5.154 feat_max=68.6
step 151 alpha=0.124 cls=0.330 dom=5.188 feat_max=75.0
step 154 alpha=0.126 cls=0.778 dom=5.563 feat_max=40.1
step 156 alp

In [17]:
import torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from common.seed import set_seed
from task2.models.backbone import ResNet18Classifier
from task2.models.domain_discriminator import DomainDiscriminator
from task2.models.grl import GradientReversal, grl_lambda
from task2.train import build_dataloaders, load_merged_config
from shared.pacs import SOURCE_DOMAINS
from shared.pacs_protocol import cyclic_loader, set_train_mode_with_frozen_bn, unpack_batch

class StableDANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.grl = GradientReversal(0.0)
        self.discriminator = DomainDiscriminator(512)
        self.lambda_domain = 1.0
        self.grl_gamma = 10.0
        self.grl_max = 1.0
    def total_loss(self, source_logits, source_labels, source_features, target_features, progress=0.0, **kw):
        alpha = grl_lambda(progress, self.grl_gamma, self.grl_max)
        self.grl.set_alpha(alpha)
        cls = F.cross_entropy(source_logits, source_labels)
        feats = torch.cat([source_features, target_features], 0)
        # L2-normalize for discriminator only
        feats_n = F.normalize(feats, p=2, dim=1)
        domain_logits = self.discriminator(self.grl(feats_n))
        domain_labels = torch.cat([
            torch.zeros(source_features.size(0), dtype=torch.long, device=feats.device),
            torch.ones(target_features.size(0), dtype=torch.long, device=feats.device),
        ])
        domain = F.cross_entropy(domain_logits, domain_labels)
        return {'loss': cls + domain, 'cls_loss': cls, 'align_loss': domain}

def run_probe(clip, normalize):
    set_seed(6304)
    cfg = load_merged_config(Path('task2/configs/base.yaml'), Path('task2/configs/dann.yaml'))
    device = torch.device('cuda')
    loaders = build_dataloaders(cfg)
    model = ResNet18Classifier().to(device)
    method = StableDANN().to(device) if normalize else None
    if not normalize:
        from task2.train import build_method
        method = build_method(cfg).to(device)
    params = list(model.parameters())+list(method.parameters())
    opt = torch.optim.AdamW(params, lr=1e-4, weight_decay=1e-4)
    src_cyc = {d: cyclic_loader(loaders['source_train'][d]) for d in SOURCE_DOMAINS}
    t_cyc = cyclic_loader(loaders['target_train'])
    steps = loaders['steps_per_epoch']
    max_dom, max_feat, last_cls = 0, 0, 0
    for step in range(202):
        progress = step/(30*steps)
        set_train_mode_with_frozen_bn(model); method.train()
        source_images, source_labels = [], []
        for d in SOURCE_DOMAINS:
            imgs, labels, _ = unpack_batch(next(src_cyc[d]), device)
            source_images.append(imgs); source_labels.append(labels)
        source_images=torch.cat(source_images); source_labels=torch.cat(source_labels)
        timgs,_,_ = unpack_batch(next(t_cyc), device)
        sf, sl = model(source_images); tf,_ = model(timgs)
        out = method.total_loss(source_logits=sl, source_labels=source_labels, source_features=sf, target_features=tf, progress=progress)
        opt.zero_grad(); out['loss'].backward()
        if clip: torch.nn.utils.clip_grad_norm_(params, clip)
        opt.step()
        max_dom=max(max_dom, out['align_loss'].item()); max_feat=max(max_feat, sf.detach().abs().max().item()); last_cls=out['cls_loss'].item()
    print(f'clip={clip} norm={normalize} max_dom={max_dom:.2f} max_feat={max_feat:.1f} end_cls={last_cls:.3f}')

run_probe(5.0, False)
run_probe(1.0, False)
run_probe(20.0, True)
run_probe(5.0, True)

clip=5.0 norm=False max_dom=37.76 max_feat=758.2 end_cls=0.567
clip=1.0 norm=False max_dom=27.00 max_feat=550.1 end_cls=1.730
clip=20.0 norm=True max_dom=0.69 max_feat=37.4 end_cls=0.392
clip=5.0 norm=True max_dom=0.69 max_feat=32.8 end_cls=0.478


In [18]:
# Wait for Drive sync of DANN L2-norm fix
from pathlib import Path
import time
for i in range(30):
    t = Path('task2/methods/dann.py').read_text()
    if 'F.normalize(feats' in t:
        print('synced after', i, 'checks')
        break
    time.sleep(2)
else:
    raise SystemExit('dann.py normalize fix not on Drive yet')
print(Path('task2/methods/cdan.py').read_text().count('normalize'), 'normalize mentions in cdan')
!python -m task2.train --method-config task2/configs/dann.yaml

synced after 0 checks
3 normalize mentions in cdan
Training dann on cuda | steps/epoch=202
[dann] epoch 01  loss=1.4018 cls=0.7614 align=0.6404  mean_src_f1=0.8484
[dann] epoch 02  loss=1.2078 cls=0.4847 align=0.7232  mean_src_f1=0.9145
[dann] epoch 03  loss=1.0951 cls=0.4047 align=0.6904  mean_src_f1=0.8978
[dann] epoch 04  loss=1.0825 cls=0.3875 align=0.6950  mean_src_f1=0.8878
[dann] epoch 05  loss=1.0238 cls=0.3364 align=0.6874  mean_src_f1=0.9089
[dann] epoch 06  loss=1.0160 cls=0.3277 align=0.6883  mean_src_f1=0.9344
[dann] epoch 07  loss=0.9745 cls=0.2874 align=0.6871  mean_src_f1=0.9227
[dann] epoch 08  loss=0.9945 cls=0.3025 align=0.6920  mean_src_f1=0.9179
[dann] epoch 09  loss=0.9783 cls=0.2816 align=0.6967  mean_src_f1=0.9354
[dann] epoch 10  loss=0.9804 cls=0.2865 align=0.6939  mean_src_f1=0.9102
[dann] epoch 11  loss=0.9464 cls=0.2535 align=0.6929  mean_src_f1=0.9321
[dann] epoch 12  loss=0.9569 cls=0.2630 align=0.6938  mean_src_f1=0.9304
[dann] epoch 13  loss=0.9629 cls=

In [19]:
!python -m task2.train --method-config task2/configs/cdan.yaml

Training cdan on cuda | steps/epoch=202
[cdan] epoch 01  loss=1.3754 cls=0.7483 align=0.6271  mean_src_f1=0.8606
[cdan] epoch 02  loss=1.1398 cls=0.4862 align=0.6536  mean_src_f1=0.8807
[cdan] epoch 03  loss=1.1465 cls=0.4304 align=0.7161  mean_src_f1=0.9095
[cdan] epoch 04  loss=1.0671 cls=0.3920 align=0.6751  mean_src_f1=0.9018
[cdan] epoch 05  loss=1.0757 cls=0.3535 align=0.7222  mean_src_f1=0.9260
[cdan] epoch 06  loss=1.0211 cls=0.3386 align=0.6824  mean_src_f1=0.9216
[cdan] epoch 07  loss=1.0098 cls=0.3144 align=0.6954  mean_src_f1=0.9066
[cdan] epoch 08  loss=1.0029 cls=0.3115 align=0.6915  mean_src_f1=0.9285
[cdan] epoch 09  loss=1.0058 cls=0.2962 align=0.7096  mean_src_f1=0.9100
[cdan] epoch 10  loss=0.9980 cls=0.3041 align=0.6939  mean_src_f1=0.9335
[cdan] epoch 11  loss=0.9850 cls=0.2929 align=0.6921  mean_src_f1=0.9410
[cdan] epoch 12  loss=0.9579 cls=0.2653 align=0.6926  mean_src_f1=0.9424
[cdan] epoch 13  loss=0.9640 cls=0.2571 align=0.7069  mean_src_f1=0.8933
[cdan] epoc

In [20]:
!python -m task2.train --method-config task2/configs/dan.yaml --lambda-mmd 0.1
!python -m task2.train --method-config task2/configs/dan.yaml --lambda-mmd 10.0

Training dan_lambda0.1 on cuda | steps/epoch=202
[dan_lambda0.1] epoch 01  loss=0.7836 cls=0.7683 align=0.1531  mean_src_f1=0.8827
[dan_lambda0.1] epoch 02  loss=0.5192 cls=0.5118 align=0.0742  mean_src_f1=0.9015
[dan_lambda0.1] epoch 03  loss=0.4250 cls=0.4191 align=0.0585  mean_src_f1=0.9167
[dan_lambda0.1] epoch 04  loss=0.3566 cls=0.3514 align=0.0516  mean_src_f1=0.9167
[dan_lambda0.1] epoch 05  loss=0.3678 cls=0.3614 align=0.0634  mean_src_f1=0.9303
[dan_lambda0.1] epoch 06  loss=0.3428 cls=0.3377 align=0.0506  mean_src_f1=0.9327
[dan_lambda0.1] epoch 07  loss=0.3353 cls=0.3310 align=0.0427  mean_src_f1=0.9362
[dan_lambda0.1] epoch 08  loss=0.2795 cls=0.2760 align=0.0350  mean_src_f1=0.9182
[dan_lambda0.1] epoch 09  loss=0.2751 cls=0.2715 align=0.0364  mean_src_f1=0.9363
[dan_lambda0.1] epoch 10  loss=0.2693 cls=0.2658 align=0.0341  mean_src_f1=0.9187
[dan_lambda0.1] epoch 11  loss=0.2701 cls=0.2666 align=0.0357  mean_src_f1=0.9388
[dan_lambda0.1] epoch 12  loss=0.2820 cls=0.2785 

In [21]:
from pathlib import Path
ckpts = [
    "task2/results/checkpoints/source_only_best.pt",
    "task2/results/checkpoints/dan_lambda1.0_best.pt",
    "task2/results/checkpoints/dann_best.pt",
    "task2/results/checkpoints/cdan_best.pt",
    "task2/results/checkpoints/dan_lambda0.1_best.pt",
    "task2/results/checkpoints/dan_lambda10.0_best.pt",
]
for p in ckpts:
    print(p, Path(p).exists(), Path(p).stat().st_size if Path(p).exists() else 0)

joined = ",".join(ckpts)
!python -m task2.evaluate_final --checkpoints {joined}

task2/results/checkpoints/source_only_best.pt True 44799051
task2/results/checkpoints/dan_lambda1.0_best.pt True 44799435
task2/results/checkpoints/dann_best.pt True 45326923
task2/results/checkpoints/cdan_best.pt True 48472651
task2/results/checkpoints/dan_lambda0.1_best.pt True 44799435
task2/results/checkpoints/dan_lambda10.0_best.pt True 44799563
Evaluating task2/results/checkpoints/source_only_best.pt ...
Evaluating task2/results/checkpoints/dan_lambda1.0_best.pt ...
Evaluating task2/results/checkpoints/dann_best.pt ...
Evaluating task2/results/checkpoints/cdan_best.pt ...
Evaluating task2/results/checkpoints/dan_lambda0.1_best.pt ...
Evaluating task2/results/checkpoints/dan_lambda10.0_best.pt ...
Wrote task2/results/tables/task2_main_comparison.json


In [22]:
import json
from pathlib import Path
d = json.loads(Path('task2/results/tables/task2_main_comparison.json').read_text())
print(f"{'run':28s} {'srcF1':>7} {'skAcc':>7} {'dAcc':>8} {'dSep':>7}")
for r in d['table']:
    dacc = r.get('target_accuracy_change_vs_source_only')
    dacc_s = f"{dacc:+.3f}" if dacc is not None else "n/a"
    print(f"{r['run']:28s} {r['mean_source_macro_f1']:7.3f} {r['target_accuracy']:7.3f} {dacc_s:>8} {r['domain_separability']:7.3f}")

# Write controlled-study slice
study = [r for r in d['table'] if r['run'].startswith('dan_lambda')]
Path('task2/results/tables/task2_lambda_study.json').write_text(json.dumps({'table': study}, indent=2))
print('wrote task2_lambda_study.json')

run                            srcF1   skAcc     dAcc    dSep
source_only_best               0.933   0.665   +0.000   0.996
dan_lambda1.0_best             0.942   0.749   +0.083   0.952
dann_best                      0.944   0.274   -0.391   0.995
cdan_best                      0.947   0.531   -0.134   0.990
dan_lambda0.1_best             0.939   0.662   -0.004   0.995
dan_lambda10.0_best            0.875   0.676   +0.011   0.930
wrote task2_lambda_study.json
